# Datalake - Jogos Olímpicos

## Abertura dos dados

In [67]:
import pandas as pd # dados
import os           # rotinas do sistema
import shutil       # arquivos e diretórios
import json

from pathlib import Path
from datetime import datetime


In [17]:
# Carregar o CSV com pandas
df_medals_historical = pd.read_csv("raw/olympics_medals_historical.csv")
df_medals_paris = pd.read_csv("raw/olympics_medals_paris2024.csv")

In [38]:
df_athletes_historical = pd.read_csv("raw/olympics_athletes_historical.csv")
df_athletes_paris = pd.read_csv("raw/olympics_athletes_paris2024.csv")


In [ ]:
df_sports_historical = pd.read_csv("raw/olympics_events_historical.csv")
df_sports_paris = pd.read_csv("raw/olympics_events_paris2024.csv")

## Definição dos metadados

In [14]:
caminho_schema = "metadata_schema.json"

metadata_schema = [
    {
        "nome" : {
            "type": "string",
            "description": "Nome do dataset"
        },
        "fonte": {
            "type": "string",
            "description": "Origem dos dados"
        },
        "descricao": {
            "type": "string",
            "description": "Explicação sobre os dados",
        }, 
        "campos": {
            "type": "array",
            "description": "Lista dos campos principais dos dados"
        },
        "data_coleta": {
            "type": "string",
            "description": "Data de criação do arquivo no formato AAAA-MM-DD"
        },
        "observacoes": {
            "type": "string",
            "description": "Detalhes adicionais sobre os dados"
        }
    }
]

with open(caminho_schema, "w", encoding="utf-8") as f:
    json.dump(metadata_schema, f, ensure_ascii=False, indent=4)



In [16]:
# Parquet
caminho_datalake = Path("raw/")
caminho_datalake.mkdir(exist_ok=True)

# Caminho do arquivo Parquet
caminho_parquet = caminho_datalake / "olympics_medals_historical.parquet"
df_medals_historical.to_parquet(caminho_parquet, engine="pyarrow", index=False)

print("Arquivo Parquet salvo em: ", caminho_parquet)

Arquivo Parquet salvo em:  raw\olympics_medals_historical.parquet


## Pré-processamento

### Medalhas

In [21]:
print(df_medals_historical.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1807 entries, 0 to 1806
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   year         1807 non-null   int64 
 1   edition      1807 non-null   object
 2   edition_id   1807 non-null   int64 
 3   country      1807 non-null   object
 4   country_noc  1807 non-null   object
 5   gold         1807 non-null   int64 
 6   silver       1807 non-null   int64 
 7   bronze       1807 non-null   int64 
 8   total        1807 non-null   int64 
dtypes: int64(6), object(3)
memory usage: 127.2+ KB
None


In [20]:
print(df_medals_paris.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   country_code  92 non-null     object
 1   country       92 non-null     object
 2   country_long  92 non-null     object
 3   Gold Medal    92 non-null     int64 
 4   Silver Medal  92 non-null     int64 
 5   Bronze Medal  92 non-null     int64 
 6   Total         92 non-null     int64 
dtypes: int64(4), object(3)
memory usage: 5.2+ KB
None


- Padronização das colunas 

In [23]:
# Base de dados 1

# Padronizando nomes dos países
country_map_1 = {
    "People's Republic of China": 'China',
    "Democratic People's Republic of Korea": 'DPR Korea',
    "Islamic Republic of Iran": 'Iran',
}

df_medals_historical['country'] = df_medals_historical['country'].replace(country_map_1)

# Separando a coluna 'edition' para "Summer Olympics" e "Winter Olympics"
df_medals_historical['edition'] = df_medals_historical['edition'].str.replace(r'\d+\s', '', regex=True)

# Verificando resultado
print(df_medals_historical.head())

# Identificando e alterando os registros da Alemanha enquanto Equipe Alemã Unida
df_medals_historical.loc[
    (df_medals_historical['country'] == "Germany") & (df_medals_historical['year'].isin([1956, 1960, 1964])),
    'country'
] = 'UT Germany'

# Verificando resultado
df_medals_historical[df_medals_historical["country"] == "UT Germany"].groupby("year").size()

   year          edition  edition_id        country country_noc  gold  silver  \
0  1896  Summer Olympics           1         Greece         GRE    10      18   
1  1900  Summer Olympics           2         France         FRA    31      41   
2  1900  Summer Olympics           2  United States         USA    20      13   
3  1904  Summer Olympics           3  United States         USA    80      85   
4  1908  Summer Olympics           5  Great Britain         GBR    56      51   

   bronze  total  
0      19     47  
1      40    112  
2      15     48  
3      83    248  
4      39    146  


year
1956    3
1960    2
1964    2
dtype: int64

In [25]:
df_medals_historical.head()

,year,edition,edition_id,country,country_noc,gold,silver,bronze,total
0,1896,Summer Olympics,1,Greece,GRE,10,18,19,47
1,1900,Summer Olympics,2,France,FRA,31,41,40,112
2,1900,Summer Olympics,2,United States,USA,20,13,15,48
3,1904,Summer Olympics,3,United States,USA,80,85,83,248
4,1908,Summer Olympics,5,Great Britain,GBR,56,51,39,146


In [ ]:
# BASE DE DADOS 2

# Padronizando nomes dos países
country_map_2 = {
    'IR Iran': 'Iran',
    'Korea': 'Republic of Korea'
}
df_medals_paris['country'] = df_medals_paris['country'].replace(country_map_2)

# Padronizando nomes das colunas
df_medals_paris['year'] = 2024
df_medals_paris['edition'] = 'Summer Olympics'
df_medals_paris['edition_id'] = 63
df_medals_paris.rename(columns={
    "country_code": "country_noc",
    "Gold Medal": "gold",
    "Silver Medal": "silver",
    "Bronze Medal": "bronze",
    "Total": "total"
    }, inplace=True)

# Removendo colunas que não serão usadas
df_medals_paris = df_medals_paris.drop(columns={"country_long"})

# Reordenando as colunas
df_medals_paris = df_medals_paris.reindex(columns=[
    'year', 'edition', 'edition_id', 'country',
    'country_noc', 'gold', 'silver', 'bronze', 'total'])

# Verificando resultado
df_medals_paris.head()

,year,edition,edition_id,country,country_noc,gold,silver,bronze,total
0,2024,Summer Olympics,63,United States,USA,40,44,42,126
1,2024,Summer Olympics,63,China,CHN,40,27,24,91
2,2024,Summer Olympics,63,Japan,JPN,20,12,13,45
3,2024,Summer Olympics,63,Australia,AUS,18,19,16,53
4,2024,Summer Olympics,63,France,FRA,16,26,22,64


### Atletas

In [73]:
df_athletes_historical.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155861 entries, 0 to 155860
Data columns (total 11 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   athlete_id     155861 non-null  int64  
 1   name           155861 non-null  object 
 2   sex            155861 non-null  object 
 3   birth_date     149211 non-null  object 
 4   birth_year     151694 non-null  float64
 5   height         105112 non-null  float64
 6   weight         104151 non-null  float64
 7   country        155861 non-null  object 
 8   country_noc    155861 non-null  object 
 9   description    54863 non-null   object 
 10  special_notes  60637 non-null   object 
dtypes: float64(3), int64(1), object(7)
memory usage: 13.1+ MB


In [74]:
df_athletes_paris.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11113 entries, 0 to 11112
Data columns (total 36 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code                11113 non-null  int64  
 1   current             11113 non-null  bool   
 2   name                11113 non-null  object 
 3   name_short          11110 non-null  object 
 4   name_tv             11110 non-null  object 
 5   gender              11113 non-null  object 
 6   function            11113 non-null  object 
 7   country_code        11113 non-null  object 
 8   country             11113 non-null  object 
 9   country_long        11113 non-null  object 
 10  nationality         11110 non-null  object 
 11  nationality_long    11110 non-null  object 
 12  nationality_code    11110 non-null  object 
 13  height              11110 non-null  float64
 14  weight              11108 non-null  float64
 15  disciplines         11113 non-null  object 
 16  even

In [33]:
print(df_athletes_paris.columns)
print(df_athletes_historical.columns)

Index(['code', 'current', 'name', 'name_short', 'name_tv', 'gender',
       'function', 'country_code', 'country', 'country_long', 'nationality',
       'nationality_long', 'nationality_code', 'height', 'weight',
       'disciplines', 'events', 'birth_date', 'birth_place', 'birth_country',
       'residence_place', 'residence_country', 'nickname', 'hobbies',
       'occupation', 'education', 'family', 'lang', 'coach', 'reason', 'hero',
       'influence', 'philosophy', 'sporting_relatives', 'ritual',
       'other_sports'],
      dtype='object')
Index(['athlete_id', 'name', 'sex', 'birth_date', 'birth_year', 'height',
       'weight', 'country', 'country_noc', 'description', 'special_notes'],
      dtype='object')


In [75]:
# Base de dados 1
rename_columns = {
    'athlete_id': 'id', 
    'sex': 'gender'
}

df_athletes_historical.rename(columns=rename_columns, inplace=True)

df_athletes_historical = df_athletes_historical[[
    "id",
    "name",
    "gender"
]]

df_athletes_historical

,id,name,gender
0,2303707,A. Sansores,Male
1,2303703,Álvaro Ledón,Male
2,921722,Abelardo Cuevas,Male
3,2303702,V. Fernández,Male
4,2303738,A. F. Ruiz,Male
...,...,...,...
155856,141672,Quan Hongchan,Female
155857,144431,Kokona Hiraki,Female
155858,141168,Rayssa Leal,Female
155859,143068,Sky Brown,Female


In [76]:
# Base de dados 2
df_athletes_paris.rename(columns={'code': 'id'}, inplace=True)

df_athletes_paris = df_athletes_paris[[
    "id",
    "name",
    "gender",
]]

df_athletes_paris

,id,name,gender
0,1532872,ALEKSANYAN Artur,Male
1,1532873,AMOYAN Malkhas,Male
2,1532874,GALSTYAN Slavik,Male
3,1532944,HARUTYUNYAN Arsen,Male
4,1532945,TEVANYAN Vazgen,Male
...,...,...,...
11108,4986655,ADA ETO Sefora,Female
11109,9460001,LIUZZI Emanuela,Female
11110,1972077,BOERS Isayah,Male
11111,1899865,STAUT Kevin,Male


In [36]:
df_athletes_historical.head()

,id,name,gender
0,2303707,A. Sansores,Male
1,2303703,Álvaro Ledón,Male
2,921722,Abelardo Cuevas,Male
3,2303702,V. Fernández,Male
4,2303738,A. F. Ruiz,Male


In [37]:
df_athletes_paris.head()

,id,name,gender
0,1532872,ALEKSANYAN Artur,Male
1,1532873,AMOYAN Malkhas,Male
2,1532874,GALSTYAN Slavik,Male
3,1532944,HARUTYUNYAN Arsen,Male
4,1532945,TEVANYAN Vazgen,Male


### Modalidades

In [57]:
df_sports_historical.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 316834 entries, 0 to 316833
Data columns (total 11 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   edition        316834 non-null  object
 1   edition_id     316834 non-null  int64 
 2   country_noc    316834 non-null  object
 3   sport          316834 non-null  object
 4   event          316834 non-null  object
 5   result_id      316834 non-null  int64 
 6   athlete        316834 non-null  object
 7   athlete_id     316834 non-null  int64 
 8   position       316834 non-null  object
 9   medal          44687 non-null   object
 10  is_team_sport  316834 non-null  bool  
dtypes: bool(1), int64(3), object(7)
memory usage: 24.5+ MB


In [58]:
df_sports_paris.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 329 entries, 0 to 328
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   event       329 non-null    object
 1   tag         329 non-null    object
 2   sport       329 non-null    object
 3   sport_code  329 non-null    object
 4   sport_url   329 non-null    object
dtypes: object(5)
memory usage: 13.0+ KB


In [59]:
df_sports_historical.head()

,edition,edition_id,country_noc,sport,event,result_id,athlete,athlete_id,position,medal,is_team_sport
0,1928 Winter Olympics,30,SUI,Skeleton,"Skeleton, Men",1,Willy von Eschen,98710,Did not finish,NaN,False
1,1928 Winter Olympics,30,FRA,Skeleton,"Skeleton, Men",1,"Jean, Comte de Beaumont",42118,Did not start,NaN,False
2,1928 Winter Olympics,30,FRA,Skeleton,"Skeleton, Men",1,Pierre Dormeuil,85267,Did not finish,NaN,False
3,1928 Winter Olympics,30,GBR,Skeleton,"Skeleton, Men",1,Lord Brabazon of Tara,1202561,Did not start,NaN,False
4,1928 Winter Olympics,30,SUI,Skeleton,"Skeleton, Men",1,Alexander Berner,84063,5,NaN,False


In [60]:
# Base de dados 1
df_sports_historical = df_sports_historical[[
    "edition",
    "edition_id",
    "sport",
    "event"
]]

df_sports_historical['year'] = df_sports_historical['edition'].str.extract(r"(\d{4})").astype(int)
df_sports_historical['edition'] = df_sports_historical['edition'].str.replace(r'\d+\s', '', regex=True)

In [61]:
df_sports_historical = df_sports_historical.reindex(columns=[
    'year', 'edition', 'edition_id', 'sport', 'event'
])
df_sports_historical.head()


,year,edition,edition_id,sport,event
0,1928,Winter Olympics,30,Skeleton,"Skeleton, Men"
1,1928,Winter Olympics,30,Skeleton,"Skeleton, Men"
2,1928,Winter Olympics,30,Skeleton,"Skeleton, Men"
3,1928,Winter Olympics,30,Skeleton,"Skeleton, Men"
4,1928,Winter Olympics,30,Skeleton,"Skeleton, Men"


In [62]:
# Base de dados 2
df_sports_paris = df_sports_paris[[
    "event",
    "sport"
]]

In [64]:
df_sports_paris['year'] = 2024
df_sports_paris['edition'] = 'Summer Olympics'
df_sports_paris['edition_id'] = 63

df_sports_paris = df_sports_paris.reindex(columns=[
    'year', 'edition', 'edition_id', 'sport', 'event'
])

df_sports_paris.head()

,year,edition,edition_id,sport,event
0,2024,Summer Olympics,63,Archery,Men's Individual
1,2024,Summer Olympics,63,Archery,Women's Individual
2,2024,Summer Olympics,63,Archery,Men's Team
3,2024,Summer Olympics,63,Archery,Women's Team
4,2024,Summer Olympics,63,Archery,Mixed Team


In [26]:
df_medals_historical.to_parquet("bronze/olympics_medals_historical.parquet", engine="pyarrow", index=False)
df_medals_paris.to_parquet("bronze/olympics_medals_paris2024.parquet", engine="pyarrow", index=False)

In [65]:
df_athletes_historical.to_parquet("bronze/olympics_athletes_historical.parquet")
df_athletes_paris.to_parquet("bronze/olympics_athletes_paris2024.parquet")

In [66]:
df_sports_historical.to_parquet("bronze/olympics_sports_historical.parquet")
df_sports_paris.to_parquet("bronze/olympics_sports_paris2024.parquet")

In [68]:
def gerar_metadata(nome, fonte, descricao, df, caminho_saida, observacoes=""):
    metadata = {
        "nome": nome,
        "fonte": fonte,
        "descricao": descricao,
        "campos": list(df.columns),
        "data_coleta": datetime.now().strftime("%d-%m-%Y"),
        "observacoes": observacoes
    }

    with open(caminho_saida, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)

## JOIN

In [71]:
df_medals = pd.concat([df_medals_historical, df_medals_paris])

df_medals

,year,edition,edition_id,country,country_noc,gold,silver,bronze,total
0,1896,Summer Olympics,1,Greece,GRE,10,18,19,47
1,1900,Summer Olympics,2,France,FRA,31,41,40,112
2,1900,Summer Olympics,2,United States,USA,20,13,15,48
3,1904,Summer Olympics,3,United States,USA,80,85,83,248
4,1908,Summer Olympics,5,Great Britain,GBR,56,51,39,146
...,...,...,...,...,...,...,...,...,...
87,2024,Summer Olympics,63,Peru,PER,0,0,1,1
88,2024,Summer Olympics,63,Qatar,QAT,0,0,1,1
89,2024,Summer Olympics,63,Singapore,SGP,0,0,1,1
90,2024,Summer Olympics,63,Slovakia,SVK,0,0,1,1


In [77]:
df_athletes = pd.concat([df_athletes_historical, df_athletes_paris])

df_athletes

,id,name,gender
0,2303707,A. Sansores,Male
1,2303703,Álvaro Ledón,Male
2,921722,Abelardo Cuevas,Male
3,2303702,V. Fernández,Male
4,2303738,A. F. Ruiz,Male
...,...,...,...
11108,4986655,ADA ETO Sefora,Female
11109,9460001,LIUZZI Emanuela,Female
11110,1972077,BOERS Isayah,Male
11111,1899865,STAUT Kevin,Male


In [78]:
df_sports = pd.concat([df_sports_historical, df_sports_paris])

df_sports

,year,edition,edition_id,sport,event
0,1928,Winter Olympics,30,Skeleton,"Skeleton, Men"
1,1928,Winter Olympics,30,Skeleton,"Skeleton, Men"
2,1928,Winter Olympics,30,Skeleton,"Skeleton, Men"
3,1928,Winter Olympics,30,Skeleton,"Skeleton, Men"
4,1928,Winter Olympics,30,Skeleton,"Skeleton, Men"
...,...,...,...,...,...
324,2024,Summer Olympics,63,Wrestling,Men's Freestyle 65kg
325,2024,Summer Olympics,63,Wrestling,Men's Freestyle 74kg
326,2024,Summer Olympics,63,Wrestling,Men's Freestyle 86kg
327,2024,Summer Olympics,63,Wrestling,Men's Freestyle 97kg


## Salvando em parquet

In [79]:
df_medals.to_parquet("bronze/olympics_medals.parquet", engine="pyarrow", index=False)
df_athletes.to_parquet("bronze/olympics_athletes.parquet", engine="pyarrow", index=False)
df_sports.to_parquet("bronze/olympics_sports.parquet", engine="pyarrow", index=False)

In [82]:
gerar_metadata(
    nome="Medalhas Olímpicas",
    fonte="Dataset Histórico + Paris 2024",
    descricao="Contagem de medalhas dos países por ano e edição",
    df=df_medals, 
    caminho_saida="bronze/olympics_medals.json",
    observacoes="Houve seleção das colunas de interesse e padronização dos nomes + concatenação"
)

In [80]:
gerar_metadata(
    nome="Atletas Olímpicos",
    fonte="Dados sobre atletas por gênero",
    descricao="Dados padronizados + Join",
    df=df_athletes, 
    caminho_saida="bronze/olympics_athletes.json",
    observacoes="Houve seleção das colunas de interesse e padronização dos nomes + concatenação"
)

In [81]:
gerar_metadata(
    nome="Jogos Olímpicos",
    fonte="Dataset Histórico + Paris 2024",
    descricao="Dados sobre as modalidades olímpicas",
    df=df_sports, 
    caminho_saida="bronze/olympics_sports.json",
    observacoes="Houve seleção das colunas de interesse e padronização dos nomes + concatenação"
)